In [1]:
import os, time, socket, threading, warnings, gc
import numpy as np, pandas as pd, torch
from joblib import Parallel, delayed
from sklearn.decomposition import FastICA
from sklearn.exceptions import ConvergenceWarning
from picard import picard
from wasserstein_ica import WassersteinICA

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(socket.gethostname(), device, torch.cuda.get_device_name(0) if device == "cuda" else "")
N_CPU = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))

OUT = "results/rebuttal_main_benchmark"; os.makedirs(OUT, exist_ok=True)
F_BASE, F_FAST, F_OT = (f"{OUT}/{n}.csv" for n in ("baselines", "fastica_runs", "otica"))
DIMS = [10, 20, 30]
POOLS = ['Continuous Only', 'Discrete Only', 'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid']
TRIALS = range(10)
N = 10_000

gpux2 cuda NVIDIA GeForce RTX 2080 Ti


In [2]:
# =============================================================================
# Amari performance index
# =============================================================================
def amari_error(W, A):
    """
    W : estimated unmixing (d×d)
    A : true mixing (d×d)
    Returns scalar ∈ [0, 1]; 0 = perfect separation.
    """
    if W is None or np.any(np.isnan(W)):
        return np.nan
    P = np.abs(W @ A)
    n = P.shape[0]
    # Sum each normalised row/column first, THEN subtract 1 once per row/column.
    # (Subtracting 1 inside np.sum would apply it to every element — d× too many.)
    row_terms = np.sum(P / np.max(P, axis=1, keepdims=True), axis=1) - 1.0
    col_terms = np.sum(P / np.max(P, axis=0, keepdims=True), axis=0) - 1.0
    return (np.sum(row_terms) + np.sum(col_terms)) / (2.0 * n)

In [3]:
# =============================================================================
# JADE (pure NumPy, no extra dependency)
# =============================================================================
def jade_ica(X):
    """
    JADE — Joint Approximate Diagonalisation of Eigenmatrices.
    Reference: Cardoso & Souloumiac, IEE Proc. F 1993.

    X   : (d, n) raw data (not pre-whitened).
    Returns W : (d, d) total unmixing such that W @ X ≈ sources.
    """
    d, n = X.shape

    # 1. Whiten
    C = X @ X.T / n
    evals, evecs = np.linalg.eigh(C)
    evals = np.maximum(evals, 1e-10)
    W_white = (1.0 / np.sqrt(evals))[:, None] * evecs.T
    Xw = W_white @ X

    # 2. Build d(d+1)/2 fourth-order cumulant matrices
    Qset = []
    for i in range(d):
        for j in range(i, d):
            xij = Xw[i] * Xw[j]
            Q   = (xij[None, :] * Xw) @ Xw.T / n
            if i == j:
                Q -= np.eye(d)
                Q[i, i] -= 2.0
            else:
                Q[i, j] -= 1.0
                Q[j, i] -= 1.0
            Qset.append((Q + Q.T) / 2.0)

    # 3. Joint diagonalisation via vectorised Jacobi sweeps (all M matrices at once)
    Qstack = np.array(Qset)   # (M, d, d)
    V = np.eye(d)

    for _sweep in range(100):
        changed = False
        for p in range(d - 1):
            for q in range(p + 1, d):
                g    = Qstack[:, p, p] - Qstack[:, q, q]
                h    = 2.0 * Qstack[:, p, q]
                ton  = float(np.dot(g, g) - np.dot(h, h))
                toff = float(2.0 * np.dot(g, h))
                denom = ton + np.sqrt(ton**2 + toff**2 + 1e-30)
                if abs(toff) < 1e-12 * (abs(denom) + 1e-12):
                    continue
                theta = 0.5 * np.arctan2(-toff, denom)
                c, s  = np.cos(theta), np.sin(theta)
                if abs(s) < 1e-12:
                    continue
                changed = True
                pr = c * Qstack[:, p, :] - s * Qstack[:, q, :]
                qr = s * Qstack[:, p, :] + c * Qstack[:, q, :]
                Qstack[:, p, :] = pr
                Qstack[:, q, :] = qr
                pc = c * Qstack[:, :, p] - s * Qstack[:, :, q]
                qc = s * Qstack[:, :, p] + c * Qstack[:, :, q]
                Qstack[:, :, p] = pc
                Qstack[:, :, q] = qc
                Vp = c * V[p, :] - s * V[q, :]
                Vq = s * V[p, :] + c * V[q, :]
                V[p, :] = Vp;  V[q, :] = Vq
        if not changed:
            break

    return V @ W_white

In [4]:
# =============================================================================
# InfoMax wrapper (via Picard)
# =============================================================================
def infomax_ica(X, n_restarts=5, seed_offset=0):
    """
    Extended InfoMax (Bell & Sejnowski 1995): unconstrained natural gradient
    (ortho=False) with extended sub/super-Gaussian density model.
    """
    for seed in range(seed_offset, seed_offset + n_restarts):
        try:
            K, W, _ = picard(
                X, fun='tanh', ortho=False, extended=True,
                max_iter=1000, tol=1e-8, random_state=seed, whiten=True
            )
            return W @ K
        except Exception:
            continue
    return None

In [5]:
# =============================================================================
# Picard ICA (orthogonal, extended)
# =============================================================================
def picard_ica(X, n_restarts=3, seed_offset=0):
    """
    Picard ICA (Ablin et al. 2018): orthogonally constrained natural gradient
    (ortho=True) with extended density model. Converges faster than sklearn
    FastICA; uses the same tanh nonlinearity, isolating contrast vs. solver.
    """
    for seed in range(seed_offset, seed_offset + n_restarts):
        try:
            K, W, _ = picard(
                X, fun='tanh', ortho=True, extended=True,
                max_iter=1000, tol=1e-8, random_state=seed, whiten=True
            )
            return W @ K
        except Exception:
            continue
    return None

In [6]:
# =============================================================================
# Data generator (exact port of ablation notebook)
# =============================================================================
def generate_mixture(n_dim, n_samples, config_type, seed=None):
    """
    config_type ∈ {'Continuous Only', 'Discrete Only',
                   'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid'}.
    """
    if seed is not None:
        np.random.seed(seed)

    def gen_laplace():     return np.random.laplace(0, 1/np.sqrt(2), n_samples)
    def gen_bernoulli():   return np.random.choice([-1., 1.], n_samples)
    def gen_uniform():     return np.random.uniform(-np.sqrt(3), np.sqrt(3), n_samples)
    def gen_student_t():   s = np.random.standard_t(3, n_samples); return s / np.std(s)
    def gen_poisson():     s = np.random.poisson(3, n_samples); return (s - s.mean()) / s.std()
    def gen_binomial():    s = np.random.binomial(10, .5, n_samples); return (s - s.mean()) / s.std()
    def gen_chisquare():   s = np.random.chisquare(2, n_samples); return (s - s.mean()) / s.std()
    def gen_exponential(): s = np.random.exponential(1, n_samples); return (s - s.mean()) / s.std()

    pools = {
        'Full Hybrid':             [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t,
                                    gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        'Continuous Only':         [gen_laplace, gen_uniform, gen_student_t,
                                    gen_chisquare, gen_exponential],
        'Discrete Only':           [gen_bernoulli, gen_poisson, gen_binomial],
        'Strictly Super-Gaussian': [gen_laplace, gen_student_t, gen_chisquare, gen_exponential],
        'Zero Gaussian':           [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t,
                                    gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        # Pure single-distribution configs (no Gaussian component)
        'Pure Laplace':            [gen_laplace],
        'Pure Uniform':            [gen_uniform],
        'Pure Student-t':          [gen_student_t],
        'Pure Bernoulli':          [gen_bernoulli],
        'Pure Poisson':            [gen_poisson],
        'Pure Binomial':           [gen_binomial],
        'Pure Chi-square':         [gen_chisquare],
        'Pure Exponential':        [gen_exponential],
    }

    active_pool = pools[config_type]
    sources = []
    is_pure = config_type == 'Zero Gaussian' or config_type.startswith('Pure ')
    if not is_pure:
        sources.append(np.random.normal(0, 1, n_samples))
        n_to_gen = n_dim - 1
    else:
        n_to_gen = n_dim

    for _ in range(n_to_gen):
        sources.append(np.random.choice(active_pool)())

    S = np.stack(sources)
    np.random.shuffle(S)

    cond_num = 1000.0
    while cond_num > 100:
        A = np.random.randn(n_dim, n_dim)
        cond_num = np.linalg.cond(A)
    return A @ S, A

In [7]:
_zq, _wq = np.polynomial.hermite_e.hermegauss(200)
C_GAUSS = float(np.sum(_wq * np.log(np.cosh(_zq))) / np.sqrt(2 * np.pi))  # E[log cosh(nu)], nu ~ N(0,1)

def negentropy_proxy(Y):
    """FastICA's own objective for components Y (d x N): sum_i (E log cosh y_i - E log cosh nu)^2."""
    Y = (Y - Y.mean(1, keepdims=True)) / Y.std(1, keepdims=True)
    return float(np.sum((np.mean(np.log(np.cosh(Y)), axis=1) - C_GAUSS) ** 2))

def append_rows(path, rows):
    pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

def done_keys(path):
    if not os.path.exists(path): return set()
    return set(map(tuple, pd.read_csv(path)[["Dim", "Config", "Trial"]].itertuples(index=False)))


In [8]:
def run_baselines(dim, trial, config):
    """FastICA (all 50 runs recorded), JADE, InfoMax, Picard on one mixture. CPU only."""
    X, A = generate_mixture(dim, N, config, seed=trial)
    key = dict(Dim=dim, Config=config, Trial=trial)
    rows, fast_rows = [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        for r in range(50):
            t0 = time.perf_counter()
            try:
                f = FastICA(n_components=dim, max_iter=10000, tol=1e-4, random_state=trial * 1000 + r).fit(X.T)
                err, J, it = amari_error(f.components_, A), negentropy_proxy(f.transform(X.T).T), f.n_iter_
            except Exception:
                err, J, it = np.nan, np.nan, np.nan
            fast_rows.append({**key, "Run": r, "Amari": err, "J": J, "n_iter": it, "Time": time.perf_counter() - t0})
    for name, fit in (("JADE", lambda: jade_ica(X)),
                      ("InfoMax", lambda: infomax_ica(X, n_restarts=5, seed_offset=trial * 17)),
                      ("Picard", lambda: picard_ica(X, n_restarts=3, seed_offset=trial * 23))):
        t0 = time.perf_counter()
        try:
            W = fit(); err = amari_error(W, A) if W is not None else np.nan
        except Exception:
            err = np.nan
        rows.append({**key, "Method": name, "Amari": err, "Time": time.perf_counter() - t0})
    return rows, fast_rows


In [9]:
def baseline_job():
    done = done_keys(F_BASE)
    todo = [(d, t, c) for d in DIMS for c in POOLS for t in TRIALS if (d, c, t) not in done]
    todo.sort(key=lambda x: (x[2] != 'Discrete Only', -x[0]))   # slowest tasks first
    for rows, fast_rows in Parallel(n_jobs=max(1, N_CPU - 1), return_as="generator_unordered")(
            delayed(run_baselines)(*x) for x in todo):
        append_rows(F_BASE, rows); append_rows(F_FAST, fast_rows)
        with open(f"{OUT}/progress_baselines.log", "a") as fh:
            fh.write(f"{time.strftime('%H:%M:%S')} d={rows[0]['Dim']} {rows[0]['Config']} trial={rows[0]['Trial']}\n")

bg = threading.Thread(target=baseline_job, daemon=True)
bg.start()


In [10]:
def run_otica(dim, trial, config):
    X, A = generate_mixture(dim, N, config, seed=trial)
    torch.manual_seed(trial)
    t0 = time.perf_counter()
    ica = WassersteinICA(torch.tensor(X, dtype=torch.float32, device=device)); ica.whiten()
    ex = []
    for _ in range(dim):
        prev = torch.stack(ex) if ex else None
        w, _ = ica.optimize_wasserstein2(prev_components=prev, max_iter=200,
                                         n_restarts=min(dim * 4, 150), dither_sigma=0.01)
        ex.append(w)
    t1 = time.perf_counter()
    W = ica.optimize_symmetric(n_components=dim, max_iter=400, lr=0.25, init_w=torch.stack(ex),
                               optimizer='stiefel', dither_sigma=0.01, batch_size=1024)
    if device == "cuda": torch.cuda.synchronize()
    t2 = time.perf_counter()
    err = amari_error(W.cpu().numpy() @ ica.W_white.cpu().numpy(), A)
    return {"Dim": dim, "Config": config, "Trial": trial, "Method": "OT-ICA", "Amari": err,
            "Time": t2 - t0, "Time_deflation": t1 - t0, "Time_stiefel": t2 - t1, "Device": device}

done = done_keys(F_OT)
for dim in DIMS:
    for config in POOLS:
        for trial in TRIALS:
            if (dim, config, trial) in done: continue
            row = run_otica(dim, trial, config)
            append_rows(F_OT, [row])
            with open(f"{OUT}/progress_otica.log", "a") as fh:
                fh.write(f"{time.strftime('%H:%M:%S')} d={dim} {config} trial={trial} amari={row['Amari']:.3f} t={row['Time']:.0f}s\n")
print("OT-ICA done | baselines still running:", bg.is_alive())


OT-ICA done | baselines still running: True


In [ ]:
print("Baseline thread still running?", bg.is_alive())

Baseline thread still running? True
